# 05 · Demo & Deploy
Launch your Career Coach LLM as a local web app and push to Hugging Face.

## 5.1 · Install Flask

In [1]:
import sys
!"{sys.executable}" -m pip install flask -q
print("✅ Flask ready")


✅ Flask ready



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 5.2 · Load the fine-tuned model
> Loads from the adapter you saved in notebook 03. Takes ~30 seconds.

In [2]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

BASE_MODEL   = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
ADAPTER_PATH = "outputs/career-coach-qlora/final-adapter"

bnb_config = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_quant_type       = "nf4",
    bnb_4bit_compute_dtype    = torch.bfloat16,
    bnb_4bit_use_double_quant = True,
)

print("Loading tokeniser...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token

print("Loading base model (4-bit)...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config = bnb_config,
    device_map          = "auto",
    torch_dtype         = torch.bfloat16,
)

print("Merging LoRA adapter...")
model = PeftModel.from_pretrained(model, ADAPTER_PATH)
model = model.merge_and_unload()
model.eval()

vram = torch.cuda.memory_reserved(0) / 1e9
print(f"\n✅ Model ready  |  VRAM used: {vram:.1f} GB")


Loading tokeniser...


c:\Users\Aman\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loading base model (4-bit)...
Merging LoRA adapter...


c:\Users\Aman\AppData\Local\Programs\Python\Python311\Lib\site-packages\peft\tuners\lora\bnb.py:325: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(



✅ Model ready  |  VRAM used: 1.1 GB


## 5.3 · Quick test before launching demo
> Verify the model actually generates a response before opening the browser.

In [3]:
def generate(instruction, input_text="", max_new_tokens=200, temperature=0.7):
    prompt = (
        f"### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n"
        if input_text.strip()
        else f"### Instruction:\n{instruction}\n\n### Response:\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        ids = model.generate(
            **inputs,
            max_new_tokens     = max_new_tokens,
            temperature        = temperature,
            top_p              = 0.9,
            do_sample          = True,
            repetition_penalty = 1.1,
            pad_token_id       = tokenizer.eos_token_id,
        )
    return tokenizer.decode(ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

# Test it
print("Testing model inference...")
response = generate("What is the STAR method for interview answers?")
print("\nModel response:")
print("-" * 50)
print(response)
print("-" * 50)
print("\n✅ Model is generating responses correctly!")


Testing model inference...

Model response:
--------------------------------------------------
The STAR method stands for "Situation, Task, Action, and Result." It is a standardized technique used to assess how well an applicant can answer job interview questions. The STAR method consists of five parts:

1. Situation: What was the situation or environment at the time of the interview?
2. Task: How did you perform the task given in the interview?
3. Action: What actions did you take during the task?
4. Result: What was the result or outcome of your performance in the task?
5. Analysis: What do you think about your performance based on this information?
--------------------------------------------------

✅ Model is generating responses correctly!


## 5.4 · Launch demo
> Opens at `http://127.0.0.1:8080` — keep this cell running while you use the app.

In [4]:
from flask import Flask, request, jsonify, render_template_string
import threading, webbrowser, time

flask_app = Flask(__name__)

HTML = """
<!DOCTYPE html>
<html>
<head>
    <title>Career Coach LLM</title>
    <style>
        * { box-sizing: border-box; margin: 0; padding: 0; }
        body { font-family: 'Segoe UI', Arial, sans-serif; max-width: 860px; margin: 0 auto; padding: 32px 20px; background: #0a0a0a; color: #f0ede8; }
        h1 { font-size: 28px; color: #e8d5a3; margin-bottom: 4px; }
        .sub { color: #666; font-size: 13px; margin-bottom: 28px; }
        .grid { display: grid; grid-template-columns: 1fr 1fr; gap: 20px; }
        label { display: block; font-size: 12px; color: #888; margin-bottom: 6px; text-transform: uppercase; letter-spacing: 0.05em; }
        textarea { width: 100%; padding: 12px; background: #161616; color: #f0ede8; border: 1px solid #2a2a2a; border-radius: 8px; font-size: 13px; resize: vertical; font-family: inherit; }
        textarea:focus { outline: none; border-color: #444; }
        .controls { display: flex; gap: 16px; margin: 12px 0; align-items: center; flex-wrap: wrap; }
        .control-group { display: flex; flex-direction: column; gap: 4px; }
        input[type=range] { width: 120px; accent-color: #e8d5a3; }
        .range-val { font-size: 12px; color: #e8d5a3; font-weight: bold; }
        button { background: #e8d5a3; color: #0a0a0a; padding: 11px 28px; border: none; border-radius: 8px; font-size: 14px; font-weight: 700; cursor: pointer; margin-top: 8px; transition: opacity 0.15s; }
        button:hover { opacity: 0.85; }
        button:disabled { opacity: 0.5; cursor: not-allowed; }
        #output-box { background: #161616; border: 1px solid #2a2a2a; border-radius: 8px; padding: 16px; min-height: 320px; }
        #output { white-space: pre-wrap; font-size: 14px; line-height: 1.7; color: #f0ede8; }
        .placeholder { color: #444; font-style: italic; }
        .generating { color: #e8d5a3; animation: pulse 1s infinite; }
        @keyframes pulse { 0%,100%{opacity:1} 50%{opacity:0.4} }
        .examples { margin-top: 20px; padding: 14px 16px; background: #111; border-radius: 8px; border: 1px solid #222; }
        .examples p { font-size: 12px; color: #666; margin-bottom: 8px; text-transform: uppercase; letter-spacing: 0.05em; }
        .ex-btn { display: inline-block; margin: 3px; padding: 5px 12px; background: #1a1a1a; border: 1px solid #2a2a2a; border-radius: 20px; font-size: 12px; color: #aaa; cursor: pointer; }
        .ex-btn:hover { background: #222; color: #e8d5a3; border-color: #444; }
        .error { color: #f87171; }
    </style>
</head>
<body>
    <h1>🎯 Career Coach LLM</h1>
    <p class="sub">Fine-tuned TinyLlama-1.1B · QLoRA on RTX 3050 · Built with Python + Hugging Face</p>

    <div class="grid">
        <div>
            <label>What do you need help with?</label>
            <textarea id="instruction" rows="3" placeholder="e.g. Review my resume summary and rewrite it to be stronger."></textarea>

            <label style="margin-top:14px">Paste resume text (optional)</label>
            <textarea id="resume" rows="7" placeholder="Paste resume content here for tailored feedback..."></textarea>

            <div class="controls">
                <div class="control-group">
                    <label>Max tokens <span class="range-val" id="tok-val">256</span></label>
                    <input type="range" id="max_tok" min="64" max="512" step="32" value="256"
                           oninput="document.getElementById('tok-val').textContent=this.value">
                </div>
                <div class="control-group">
                    <label>Temperature <span class="range-val" id="temp-val">0.7</span></label>
                    <input type="range" id="temp" min="0.1" max="1.0" step="0.05" value="0.7"
                           oninput="document.getElementById('temp-val').textContent=parseFloat(this.value).toFixed(2)">
                </div>
            </div>
            <button id="btn" onclick="getAdvice()">Get Career Advice →</button>
        </div>

        <div>
            <label>Response</label>
            <div id="output-box">
                <div id="output" class="placeholder">Your career coaching advice will appear here...</div>
            </div>
        </div>
    </div>

    <div class="examples">
        <p>Quick examples</p>
        <span class="ex-btn" onclick="setExample('Review my resume summary and rewrite it.', 'Hardworking developer with 3 years experience seeking new opportunities in tech.')">Rewrite summary</span>
        <span class="ex-btn" onclick="setExample('What keywords is this resume missing for a backend engineering role?', 'Skills: Python, Django, MySQL. Built internal tools at a startup.')">Keyword gaps</span>
        <span class="ex-btn" onclick="setExample('Rewrite this bullet point to show measurable impact.', 'Responsible for improving the website performance.')">Improve bullet</span>
        <span class="ex-btn" onclick="setExample('What is the STAR method for interview answers?', '')">STAR method</span>
        <span class="ex-btn" onclick="setExample('How do I negotiate a salary offer of ₹12 LPA in Bangalore?', '')">Salary negotiation</span>
        <span class="ex-btn" onclick="setExample('How should I format my resume for ATS systems?', '')">ATS formatting</span>
    </div>

    <script>
        function setExample(instruction, resume) {
            document.getElementById('instruction').value = instruction;
            document.getElementById('resume').value = resume;
        }

        async function getAdvice() {
            const btn = document.getElementById('btn');
            const out = document.getElementById('output');
            const instruction = document.getElementById('instruction').value.trim();

            if (!instruction) {
                out.className = 'error';
                out.textContent = 'Please enter a question or instruction first.';
                return;
            }

            btn.disabled = true;
            btn.textContent = 'Generating...';
            out.className = 'generating';
            out.textContent = 'Thinking...';

            try {
                const res = await fetch('/generate', {
                    method: 'POST',
                    headers: {'Content-Type': 'application/json'},
                    body: JSON.stringify({
                        instruction: instruction,
                        resume: document.getElementById('resume').value,
                        max_tokens: parseInt(document.getElementById('max_tok').value),
                        temperature: parseFloat(document.getElementById('temp').value)
                    })
                });
                const data = await res.json();
                out.className = '';
                out.textContent = data.response || 'No response generated.';
            } catch(e) {
                out.className = 'error';
                out.textContent = 'Error: ' + e.message;
            }

            btn.disabled = false;
            btn.textContent = 'Get Career Advice →';
        }

        // Allow Enter key in instruction box
        document.addEventListener('DOMContentLoaded', () => {
            document.getElementById('instruction').addEventListener('keydown', e => {
                if (e.key === 'Enter' && e.ctrlKey) getAdvice();
            });
        });
    </script>
</body>
</html>
"""

@flask_app.route("/")
def index():
    return render_template_string(HTML)

@flask_app.route("/generate", methods=["POST"])
def generate_endpoint():
    try:
        data        = request.get_json(force=True)
        instruction = data.get("instruction", "").strip()
        resume_text = data.get("resume", "").strip()
        max_tokens  = min(int(data.get("max_tokens", 256)), 512)
        temperature = float(data.get("temperature", 0.7))

        if not instruction:
            return jsonify({"response": "Please enter an instruction."})

        response = generate(instruction, resume_text, max_tokens, temperature)
        return jsonify({"response": response})

    except Exception as e:
        return jsonify({"response": f"Error: {str(e)}"}), 500

PORT = 8080

def run_server():
    flask_app.run(host="127.0.0.1", port=PORT, debug=False,
                  use_reloader=False, threaded=True)

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(1.5)   # wait for server to start

print(f"✅ Demo running at: http://127.0.0.1:{PORT}")
print("   Ctrl+Enter inside the instruction box to submit")
print("   Keep this cell running — interrupt to stop the server")
webbrowser.open(f"http://127.0.0.1:{PORT}")


 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:8080
Press CTRL+C to quit


✅ Demo running at: http://127.0.0.1:8080
   Ctrl+Enter inside the instruction box to submit
   Keep this cell running — interrupt to stop the server


True

## 5.5 · Push to Hugging Face Hub
> Get a permanent public URL for your portfolio.

In [1]:
from huggingface_hub import login
login()   # paste your token from huggingface.co/settings/tokens


In [4]:
HF_REPO_ID = "AmanT420/career-coach-llm"   # ← change this!

from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
)
push_tok   = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")
push_model = AutoModelForCausalLM.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    quantization_config=bnb_config, device_map="auto", torch_dtype=torch.bfloat16,
)
push_model = PeftModel.from_pretrained(push_model, r"D:/career-coach-notebooks/outputs/career-coach-qlora/final-adapter")
push_model.push_to_hub(HF_REPO_ID)
push_tok.push_to_hub(HF_REPO_ID)
print(f"\n✅ Model live at: https://huggingface.co/{HF_REPO_ID}")


README.md: 0.00B [00:00, ?B/s]

c:\Users\Aman\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:157: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Aman\.cache\huggingface\hub\models--AmanT420--career-coach-llm. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to see activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


README.md: 0.00B [00:00, ?B/s]


✅ Model live at: https://huggingface.co/AmanT420/career-coach-llm


In [5]:
import os, shutil

# Move data and outputs up to career-coach-llm/
root = os.path.abspath(os.path.join(os.getcwd(), ".."))  # parent of notebooks/
print(f"Project root: {root}")

for folder in ["data", "outputs"]:
    src = os.path.join(os.getcwd(), folder)
    dst = os.path.join(root, folder)
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.move(src, dst)
        print(f"✅ Moved {folder}/ → project root")
    elif os.path.exists(dst):
        print(f"✅ {folder}/ already in project root")
    else:
        print(f"⚠️  {folder}/ not found")

Project root: d:\career-coach-notebooks
✅ data/ already in project root
✅ outputs/ already in project root


In [6]:
# Create README.md in project root
readme = """# Career Coach LLM

Fine-tuned TinyLlama-1.1B for career coaching using QLoRA (4-bit NF4).
Trained on RTX 3050 consumer GPU.

## Results
| Metric   | Base Model | Fine-tuned | Improvement |
|----------|-----------|------------|-------------|
| ROUGE-L  | 0.18      | X.XX       | +XX%        |

## Tech Stack
- Model: TinyLlama-1.1B-Chat-v1.0
- Method: QLoRA (PEFT, bitsandbytes 4-bit NF4)
- LoRA rank: r=8, alpha=16
- Hardware: NVIDIA RTX 3050
- Demo: Flask web app

## Run
```bash
pip install -r requirements.txt
jupyter notebook notebooks/
```

## Resume Bullet
Career Coach LLM | Python · QLoRA · TinyLlama · Hugging Face
- Fine-tuned TinyLlama-1.1B on career-coaching dataset using QLoRA on RTX 3050
- Improved ROUGE-L by XX% over base model
- Deployed as live web demo
"""

requirements = """torch==2.2.0
transformers==4.40.0
peft==0.10.0
bitsandbytes==0.43.1
accelerate==0.29.3
datasets==2.19.0
trl==0.8.6
rouge-score==0.1.2
flask
scipy
sentencepiece
"""

root = os.path.abspath(os.path.join(os.getcwd(), ".."))

with open(os.path.join(root, "README.md"), "w") as f:
    f.write(readme)
print("✅ README.md created")

with open(os.path.join(root, "requirements.txt"), "w") as f:
    f.write(requirements)
print("✅ requirements.txt created")

with open(os.path.join(root, ".gitignore"), "w") as f:
    f.write("__pycache__/\n*.pyc\n.ipynb_checkpoints/\noutputs/\n*.dll\n")
print("✅ .gitignore created")

✅ README.md created
✅ requirements.txt created
✅ .gitignore created


In [ ]:
# ── CareerMind: Push adapter to HuggingFace Hub ──────────────────────
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from huggingface_hub import login

# Step 1: Login — a text box will appear, paste your hf_ Write token
login()

# Step 2: Your details (already filled in!)
HF_REPO_ID   = "AmanT420/career-coach-llm"
BASE_MODEL   = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
ADAPTER_PATH = r"D:\career-coach-notebooks\outputs\career-coach-qlora\final-adapter"

# Step 3: Load tokenizer
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token

# Step 4: Load base model in 4-bit
print("Loading base model (~30 sec)...")
bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_cfg,
    device_map="auto",
    trust_remote_code=True,
)

# Step 5: Load your LoRA adapter
print("Loading LoRA adapter from D: drive...")
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)

# Step 6: Push — only uploads ~40-80 MB adapter, not the 2 GB base model
print("Pushing adapter to HF Hub...")
model.push_to_hub(HF_REPO_ID)

print("Pushing tokenizer...")
tokenizer.push_to_hub(HF_REPO_ID)

print("\n✓ Done! Check: https://huggingface.co/AmanT420/career-coach-llm")

Loading tokenizer...
Loading base model (~30 sec)...
Loading LoRA adapter from D: drive...


ValueError: Can't find 'adapter_config.json' at 'D:\career-coach-notebooks\notebooks\outputs\career-coach-qlora\final-adapter'

## 5.6 · Resume bullet

```
Career Coach LLM  |  Python · QLoRA · TinyLlama · Hugging Face  |  github.com/you/career-coach-llm
• Fine-tuned TinyLlama-1.1B on a curated career-coaching dataset using QLoRA (4-bit NF4,
  LoRA r=8) on a consumer RTX 3050 GPU; improved ROUGE-L by X% over the base model
• Built a Flask web demo with real-time inference; deployed adapter to Hugging Face Hub
```
> Replace X% with your actual score from notebook 04.
